# SC2Artic-Q11

Analyzes SARS-CoV-2 sequence data from Q11 sequencing run (Illumina MiSeq 500 cycle nano kit).


## Setup and Alignment Commands
Define the file paths and perform alignment using BWA to the SARS-CoV-2 reference genome.

In [ ]:
import os
import subprocess
import glob
import pandas as pd
import re
from IPython.display import display

# Safely identify the original notebook project root even if the kernel has already been subjected to %cd shifts
if 'notebook_dir' not in globals() or not notebook_dir.endswith('RespiCov-analyze'):
    # Search backwards until we hit the correct project root
    current = os.path.abspath(os.curdir)
    while current and not current.endswith('RespiCov-analyze') and current != '/':
        current = os.path.dirname(current)
    notebook_dir = current if current.endswith('RespiCov-analyze') else os.path.abspath(os.curdir)
    
    work_dir = os.path.join(notebook_dir, "../Q11-MG1-SC2")
    out_dir = os.path.join(work_dir, "alignments")
    import shutil
    if os.path.exists(out_dir):
        shutil.rmtree(out_dir)
    os.makedirs(out_dir, exist_ok=True)
    get_ipython().run_line_magic("cd", out_dir)

# Define pipeline execution variables relative to the securely locked origin path
fastq_dir = os.path.join(work_dir, "JobID-872")
reference = os.path.abspath(os.path.join(notebook_dir, "../primer_schemes/SARS-CoV-2.reference.fasta"))
scheme = os.path.abspath(os.path.join(notebook_dir, "../primer_schemes/ARTIC-V5.4.2.scheme.bed"))
q10_out_dir = os.path.join(work_dir, "alignments_q10")

samples = ["RB1-S116", "RB2-S151", "RB3-S171"]

if not os.path.exists(reference + ".bwt"):
    print("Indexing reference genome...")
    subprocess.run(["bwa", "index", reference], check=True)
else:
    print("Reference genome already indexed.")

def align_and_trim(sample, q_threshold=20, target_out_dir=out_dir):
    print(f"Aligning {sample}...")
    r1_files = glob.glob(os.path.join(fastq_dir, f"{sample}_*_R1_001.fastq.gz"))
    r2_files = glob.glob(os.path.join(fastq_dir, f"{sample}_*_R2_001.fastq.gz"))
    
    if not r1_files or not r2_files:
        print(f"Could not find reads for {sample}, skipping...")
        return None
        
    r1, r2 = r1_files[0], r2_files[0]
    bam_out = os.path.join(target_out_dir, f"{sample}.bam")
    trim_prefix = os.path.join(target_out_dir, f"{sample}.trimmed")
    trim_bam = trim_prefix + ".bam"
    final_bam = os.path.join(target_out_dir, f"{sample}.trimmed.sorted.bam")
    
    bwa_cmd = f"bwa mem -t 4 {reference} {r1} {r2} | samtools sort -o {bam_out}"
    subprocess.run(bwa_cmd, shell=True, check=True)
    subprocess.run(["samtools", "index", bam_out], check=True)
    
    print(f"Trimming primers for {sample}...")
    ivar_cmd = f"ivar trim -b {scheme} -q {q_threshold} -i {bam_out} -p {trim_prefix}"
    res = subprocess.run(ivar_cmd, shell=True, capture_output=True, text=True)
    
    if res.returncode != 0:
        print(f"ivar trim failed for {sample}: {res.stderr}")
    else:
        print(res.stderr)
        
    if os.path.exists(trim_bam):
        sort_cmd = f"samtools sort -o {final_bam} {trim_bam}"
        subprocess.run(sort_cmd, shell=True, check=True)
        subprocess.run(["samtools", "index", final_bam], check=True)
        os.remove(trim_bam)
        
    stats = {"Sample": sample}
    primer_counts = {}
    lines = res.stdout.split('\n') + res.stderr.split('\n')
    parsing_primers = False
    
    for line in lines:
        line = line.strip()
        if line.startswith("Primer Name\tRead Count"):
            parsing_primers = True
            continue
        if parsing_primers:
            if not line or line.startswith("Trimmed primers") or line.startswith("-------"):
                parsing_primers = False
            else:
                parts = line.split('\t')
                if len(parts) == 2:
                    primer_counts[parts[0]] = int(parts[1])
                    
        m1 = re.search(r'Trimmed primers from.*?\((\d+)\)', line)
        if m1: stats["Primer Trimmed (reads)"] = int(m1.group(1))
        
        m2 = re.search(r'.*?\((\d+)\) of reads were quality trimmed', line)
        if m2: stats["Quality Trimmed (reads)"] = int(m2.group(1))
        
        m3 = re.search(r'.*?\((\d+)\) of reads that started outside of primer regions', line)
        if m3: stats["Outside Primer Regions (reads)"] = int(m3.group(1))
        
    stats["Primer_Counts"] = primer_counts
    return stats


def generate_consensus(sample, target_out_dir=out_dir):
    print(f"Generating consensus for {sample}...")
    final_bam = os.path.join(target_out_dir, f"{sample}.trimmed.sorted.bam")
    consensus_prefix = os.path.join(target_out_dir, f"{sample}.consensus")
    fasta_out = consensus_prefix + ".fa"
    
    if not os.path.exists(final_bam):
        print(f"BAM file not found for {sample}")
        return None
        
    mpileup_cmd = f"samtools mpileup -aa -A -d 0 -B -Q 0 --reference {reference} {final_bam}"
    ivar_cmd = f"ivar consensus -p {consensus_prefix} -m 10 -n N"
    
    full_cmd = f"{mpileup_cmd} | {ivar_cmd}"
    subprocess.run(full_cmd, shell=True, check=True)
    
    if os.path.exists(fasta_out):
        with open(fasta_out, 'r') as f:
            lines = f.readlines()
            
        seq = "".join([line.strip() for line in lines if not line.startswith(">")])
        total_len = len(seq)
        n_count = seq.upper().count('N')
        coverage = ((total_len - n_count) / total_len) if total_len > 0 else 0
        
        return {
            "Sample": sample,
            "Total Length": total_len,
            "N Count": n_count,
            "Coverage": round(coverage, 4)
        }
    return None

## Base Quality Distribution
Visualize the Phred base quality scores of the input reads using Biopython. In Illumina sequencing we expect quality to start high and drop off as the read progresses, and also R2 to have somewhat lower quality than R1 (since it occurs afterwards).

In [ ]:
import gzip
import glob
import matplotlib.pyplot as plt
from Bio import SeqIO
from collections import Counter

for sample in samples:
    r1_files = glob.glob(os.path.join(fastq_dir, f"{sample}_*_R1_001.fastq.gz"))
    r2_files = glob.glob(os.path.join(fastq_dir, f"{sample}_*_R2_001.fastq.gz"))
    
    if not r1_files or not r2_files: 
        continue
        
    def get_qualities(fastq_file):
        dist_counts = Counter()
        pos_sums = []
        pos_counts = []
        pos_q20_counts = []
        
        with gzip.open(fastq_file, "rt") as handle:
            for record in SeqIO.parse(handle, "fastq"):
                quals = record.letter_annotations["phred_quality"]
                
                # 1. Total Distribution Tracking
                dist_counts.update(quals)
                
                # 2. Positional Tracking
                while len(pos_sums) < len(quals):
                    pos_sums.append(0)
                    pos_counts.append(0)
                    pos_q20_counts.append(0)
                for i, q in enumerate(quals):
                    pos_sums[i] += q
                    pos_counts[i] += 1
                    if q >= 20:
                        pos_q20_counts[i] += 1
                    
        avg_quals = [s/c if c > 0 else 0 for s, c in zip(pos_sums, pos_counts)]
        frac_q20 = [q/c if c > 0 else 0 for q, c in zip(pos_q20_counts, pos_counts)]
        return dist_counts, avg_quals, frac_q20

    print(f"Processing fastq qualities for {sample}...")
    r1_dist, r1_pos, r1_q20 = get_qualities(r1_files[0])
    r2_dist, r2_pos, r2_q20 = get_qualities(r2_files[0])
    
    scores = list(range(51))
    r1_freqs = [r1_dist.get(s, 0) for s in scores]
    r2_freqs = [r2_dist.get(s, 0) for s in scores]
            
    if sum(r1_freqs) > 0 or sum(r2_freqs) > 0:
        fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(24, 5))
        
        # Plot 1: Overall Quality Distribution
        ax1.plot(scores, r1_freqs, label='R1', color='mediumpurple', linewidth=2)
        ax1.plot(scores, r2_freqs, label='R2', color='darkorange', linewidth=2)
        ax1.set_title(f"Base Quality Distribution - {sample}", fontsize=14)
        ax1.set_xlabel("Phred Quality Score", fontsize=12)
        ax1.set_ylabel("Frequency", fontsize=12)
        ax1.set_xlim(0, 50)
        ax1.legend()
        
        # Plot 2: Average Quality By Read Position
        ax2.plot(range(1, len(r1_pos) + 1), r1_pos, label='R1', color='mediumpurple', linewidth=2)
        ax2.plot(range(1, len(r2_pos) + 1), r2_pos, label='R2', color='darkorange', linewidth=2)
        ax2.set_title(f"Average Quality by Read Position - {sample}", fontsize=14)
        ax2.set_xlabel("Read Position (bp)", fontsize=12)
        ax2.set_ylabel("Average Phred Score", fontsize=12)
        ax2.set_ylim(0, 50)
        ax2.legend()
        
        # Plot 3: Fraction >= Q20 By Position
        ax3.plot(range(1, len(r1_q20) + 1), r1_q20, label='R1', color='mediumpurple', linewidth=2)
        ax3.plot(range(1, len(r2_q20) + 1), r2_q20, label='R2', color='darkorange', linewidth=2)
        ax3.set_title(f"Fraction >= Q20 by Position - {sample}", fontsize=14)
        ax3.set_xlabel("Read Position (bp)", fontsize=12)
        ax3.set_ylabel("Fraction >= Q20", fontsize=12)
        ax3.set_ylim(0, 1.05)
        ax3.legend()
        
        plt.tight_layout()
        plt.show()

In [ ]:
stats_RB1_S116 = align_and_trim("RB1-S116")

In [ ]:
stats_RB2_S151 = align_and_trim("RB2-S151")

In [ ]:
stats_RB3_S171 = align_and_trim("RB3-S171")

## Alignment Statistics
Calculate and display the number and fraction of mapped reads for each sample using `samtools flagstat` and `ivar trim` logs.

In [ ]:
import pandas as pd\n\nresults = []

for sample in samples:
    raw_bam = os.path.join(out_dir, f"{sample}.bam")
    trim_bam = os.path.join(out_dir, f"{sample}.trimmed.sorted.bam")
    
    if not os.path.exists(raw_bam): continue
        
    res_raw = subprocess.run(["samtools", "flagstat", raw_bam], capture_output=True, text=True, check=True)
    total_reads = 0
    mapped_reads = 0
    for line in res_raw.stdout.split('\n'):
        if "in total" in line: total_reads = int(line.split(' + ')[0])
        elif "mapped (" in line: mapped_reads = int(line.split(' + ')[0]); break
            
    trimmed_reads = 0
    if os.path.exists(trim_bam):
        res_trim = subprocess.run(["samtools", "flagstat", trim_bam], capture_output=True, text=True, check=True)
        for line in res_trim.stdout.split('\n'):
            if "mapped (" in line: trimmed_reads = int(line.split(' + ')[0]); break
                
    total = total_reads
    unmapped = total - mapped_reads
    unmapped_str = f"{unmapped} ({(unmapped/total):.1%})" if total > 0 else "0 (0%)"
    
    row = {
        "Sample": sample,
        "Total Reads": total,
        "Unmapped": unmapped_str,
        "Mapped Reads": mapped_reads,
        "Clipped Mapped Reads": trimmed_reads
    }
    results.append(row)

pd.set_option('display.max_rows', None)
df_results = pd.DataFrame(results)
display(df_results)\n\nprint('\n--- Consensus Statistics ---')\ndisplay(pd.DataFrame(consensus_stats))

## Primer Coverage Statistics
Generate a table and bar chart showing reads per primer based on `ivar trim` output.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import re

primer_data = []

# Build dataset directly from captured ivar primer counts
for sample in samples:
    if sample in trim_stats_dict and "Primer_Counts" in trim_stats_dict[sample]:
        for primer, count in trim_stats_dict[sample]["Primer_Counts"].items():
            match = re.search(r'SARS-CoV-2_(\d+)_(LEFT|RIGHT)', primer)
            if match:
                num = int(match.group(1))
                direction = match.group(2)[0]
                locus = f"{num}{direction}"
                primer_data.append({
                    "Sample": sample,
                    "Primer_Num": num,
                    "Direction": direction,
                    "Locus": locus,
                    "Read Count": count
                })

df_primers = pd.DataFrame(primer_data)

if not df_primers.empty:
    # Group and sum the Read Counts by Sample and Locus to combine variants (e.g. 84_RIGHT_3 and 84_RIGHT_2)
    df_grouped = df_primers.groupby(["Sample", "Primer_Num", "Direction", "Locus"])["Read Count"].sum().reset_index()
    
    # Display Table showing the summed locus counts
    df_pivot_table = df_grouped.pivot(index="Locus", columns="Sample", values="Read Count").fillna(0)
    
    def locus_sort_key(locus):
        match = re.match(r'(\d+)([LR])', locus)
        if match:
            return (int(match.group(1)), match.group(2))
        return (0, locus)
        
    df_pivot_table['sort_key'] = [locus_sort_key(idx) for idx in df_pivot_table.index]
    df_pivot_table = df_pivot_table.sort_values('sort_key').drop('sort_key', axis=1)
    
    with pd.option_context('display.max_rows', None):
        display(df_pivot_table)
        
    # Draw grouped numeric Bar Charts
    for sample in samples:
        sample_data = df_grouped[df_grouped["Sample"] == sample]
        if sample_data.empty:
            continue
            
        df_sample_pivot = sample_data.pivot(index="Primer_Num", columns="Direction", values="Read Count").fillna(0)
        
        fig, ax = plt.subplots(figsize=(20, 5))
        
        if 'L' in df_sample_pivot.columns:
            ax.bar(df_sample_pivot.index - 0.2, df_sample_pivot['L'], width=0.4, color='steelblue', label='L')
        if 'R' in df_sample_pivot.columns:
            ax.bar(df_sample_pivot.index + 0.2, df_sample_pivot['R'], width=0.4, color='darkorange', label='R')
            
        ax.set_title(f"Reads per Primer - {sample}", fontsize=16)
        ax.set_xlabel("Primer Number", fontsize=12)
        ax.set_ylabel("Read Count", fontsize=12)
        ax.legend(title="Direction")
        
        plt.tight_layout()
        plt.show()

else:
    print("No primer depth data found from ivar output.")

## Consensus Sequence
Generate a consensus sequence for each sample using `samtools mpileup` and `ivar consensus`, then calculate the frequency of 'N's.

In [ ]:
cons_RB1_S116 = generate_consensus("RB1-S116")

In [ ]:
cons_RB2_S151 = generate_consensus("RB2-S151")

In [ ]:
cons_RB3_S171 = generate_consensus("RB3-S171")

### Consensus Statistics

In [ ]:
consensus_stats = [s for s in [cons_RB1_S116, cons_RB2_S151, cons_RB3_S171] if s]
df_consensus = pd.DataFrame(consensus_stats)
display(df_consensus)

## Nextclade Analysis
Generate a Nextclade report for the 3 generated consensus sequences.

In [ ]:
import subprocess
import pandas as pd
import os
from IPython.display import display, HTML

# Switch context into the primary output directory 
os.chdir(out_dir)

combined_fasta = os.path.join(out_dir, "all_consensus.fa")
with open(combined_fasta, "w") as outfile:
    for sample in samples:
        fasta_in = os.path.join(out_dir, f"{sample}.consensus.fa")
        if os.path.exists(fasta_in):
            with open(fasta_in, "r") as infile:
                lines = infile.readlines()
                outfile.write(f">{sample}\n")
                for line in lines:
                    if not line.startswith(">"):
                        outfile.write(line)

nextclade_out = os.path.join(out_dir, "nextclade_report.tsv")
nextclade_html = os.path.join(out_dir, "nextclade_report.html")
dataset_dir = os.path.join(out_dir, "sars-cov-2-dataset")

print("Running Nextclade on consensus sequences...")

try:
    if not os.path.exists(dataset_dir):
        subprocess.run(["nextclade", "dataset", "get", "--name", "sars-cov-2", "--output-dir", dataset_dir], check=True)
    subprocess.run(["nextclade", "run", "--input-dataset", dataset_dir, "--output-tsv", nextclade_out, combined_fasta], check=True)
except subprocess.CalledProcessError as e:
    print("Nextclade execution failed:", e)
except FileNotFoundError:
    print("Nextclade tool not found in PATH.")

if os.path.exists(nextclade_out):
    df_nc = pd.read_csv(nextclade_out, sep='\t')
    
    # Compute relative substitutions and deletions from Pango founder
    def count_muts(mut_str):
        if pd.isna(mut_str): return 0
        s = str(mut_str).strip()
        return len(s.split(',')) if s else 0
        
    df_nc["pango_substitutions"] = df_nc.get("privateNucMutations.totalPrivateSubstitutions", df_nc["totalSubstitutions"])
    
    if "founderMuts['Nextclade_pango'].deletions" in df_nc.columns:
        df_nc["pango_deletions"] = df_nc["totalDeletions"] - df_nc["founderMuts['Nextclade_pango'].deletions"].apply(count_muts)
    else:
        df_nc["pango_deletions"] = df_nc["totalDeletions"]
        
    df_nc["pango_insertions"] = df_nc["totalInsertions"] # Insertions are extremely rare as founder mutations
    
    cols_to_show = ["seqName", "clade", "Nextclade_pango", "pango_substitutions", "pango_deletions", "pango_insertions", "totalMissing", "qc.overallScore", "qc.overallStatus"]
    
    df_display = df_nc[[c for c in cols_to_show if c in df_nc.columns]].rename(columns={
        "pango_substitutions": "Pango Relative Substitutions",
        "pango_deletions": "Pango Relative Deletions",
        "pango_insertions": "Pango Relative Insertions"
    })
    display(df_display)




## Q15 Threshold Analysis
Repeats the entire sequence analysis using an `ivar trim` quality threshold of Q15 (instead of the default Q20).